# Day 26 — Bivariate Analysis and Correlation

This notebook covers Covariance, Pearson Correlation Coefficient ($r$), Spearman Rank Correlation ($\rho$), Outlier Distortion, and Multicollinearity

## Validation goals
- Distinguish between covariance and correlation, highlighting why covariance is scale-dependent
- Calculate and interpret the Pearson correlation coefficient ($r$) for linear relationships
- Apply Spearman rank correlation ($\rho$) to capture non-linear monotonic relationships and resist outliers
- Analyze the vulnerability of Pearson $r$ to extreme leverage points and outliers
- Evaluate the impact of perfect multicollinearity ($r = 1.0$) on Ordinary Least Squares (OLS) regression matrix invertibility

Up to this point, Week 4 has focused on univariate analysis—examining the properties of a single variable in isolation (mean, variance, percentiles, and distribution shape). However, real-world data science and machine learning tasks almost always involve studying how multiple variables interact with one another.

Bivariate analysis measures the statistical relationship between two variables. Correlation specifically quantifies the strength and direction of association between them, providing a vital tool for feature selection, multicollinearity detection, and exploratory data analysis.

### Core Concepts & Deep Dive1.

**Covariance: The Direction of Association**
Before defining correlation, we must look at covariance, which measures how two variables vary together:$$\text{Cov}(X, Y) = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{n - 1}$$

- Interpretation:
    - If $X$ and $Y$ tend to increase together, their deviations are positive on average, resulting in a positive covariance.
    - If one variable increases as the other decreases, the covariance is negative.

- The Scale Limitation: Covariance has no upper or lower bound; its magnitude depends entirely on the units of measurement (e.g., measuring revenue in dollars versus thousands of dollars changes the covariance value dramatically). This makes it impossible to compare the strength of relationships across different datasets.

2. **Pearson Correlation Coefficient ($r$)**

To solve covariance's scale dependency, we standardize the value by dividing by the product of the standard deviations of both variables. This yields the Pearson correlation coefficient ($r$), which ranges strictly from $-1$ to $+1$:$$r_{XY} = \frac{\text{Cov}(X, Y)}{s_x s_y} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2 \sum_{i=1}^{n} (y_i - \bar{y})^2}}$$

- Interpretation of Values:$r = +1$: 

    - Perfect positive linear relationship.
    - $r = 0$: No linear relationship.
    - $r = -1$: Perfect negative linear relationship.

- Assumptions: Pearson correlation assumes both variables are continuous, approximately normally distributed, and linearly related. It is heavily distorted by extreme outliers.

3. **Spearman Rank Correlation ($\rho$)**

When relationships are non-linear but monotonic (consistently increasing or decreasing), or when data contains severe outliers and non-normal distributions, the Spearman rank correlation coefficient ($\rho$) is preferred.
- Mechanism: Instead of evaluating raw values, Spearman converts both variables into ranks (sorting values from smallest to largest and assigning integer ranks) and then calculates the Pearson correlation on those ranks.
- Robustness: Because it relies on ranks rather than absolute values, it is robust to outliers and can capture non-linear monotonic trends.

4. **The Pitfall of Correlation vs. Causation**
A foundational rule in data science: Correlation does not imply causation.

High correlation between two variables can be driven by a confounding variable (a third unobserved factor influencing both), or it can be entirely coincidental (spurious correlation).

### Practical Implementation: Calculating Correlation in Python

Here is a script demonstrating how to compute Pearson and Spearman correlation matrices using Pandas, and how outliers can distort linear correlation.

In [1]:
import numpy as np
import pandas as pd

# 1. Generate Synthetic Bivariate Data (Study Hours vs Exam Scores)
np.random.seed(42)
study_hours = np.random.uniform(1.0, 10.0, size=100)
# Perfect linear relationship plus minor noise
exam_scores = 10 * study_hours + 15 + np.random.normal(0, 3, size=100)

df = pd.DataFrame({
    "study_hours": study_hours,
    "exam_scores": exam_scores
})

# Introduce a non-linear monotonic relationship column (exponential growth)
df["practice_score"] = np.exp(study_hours / 3.0)

# ==========================================
# 2. Computing Correlation Matrices
# ==========================================
# Pearson correlation (measures linear association)
pearson_corr = df.corr(method="pearson")

# Spearman correlation (measures monotonic rank association)
spearman_corr = df.corr(method="spearman")

print("--- Pearson Correlation Matrix ---")
print(pearson_corr.round(3))

print("\n--- Spearman Correlation Matrix ---")
print(spearman_corr.round(3))

--- Pearson Correlation Matrix ---
                study_hours  exam_scores  practice_score
study_hours           1.000        0.995           0.937
exam_scores           0.995        1.000           0.938
practice_score        0.937        0.938           1.000

--- Spearman Correlation Matrix ---
                study_hours  exam_scores  practice_score
study_hours           1.000        0.994           1.000
exam_scores           0.994        1.000           0.994
practice_score        1.000        0.994           1.000


### Guided Question:

Suppose you analyze the relationship between website traffic and server response latency. As traffic increases linearly up to a threshold, latency remains flat, but past that threshold, latency spikes exponentially.

Why might say the Pearson correlation coefficient fail to capture this strong relationship accurately, whereas the Spearman rank correlation succeeds?

### Answer:

The fundamental difference between the Pearson and Spearman correlation coefficients lies in what type of relationship they are designed to detect: Pearson measures linear association, while Spearman measures monotonic association (whether variables tend to move in the same relative direction, regardless of the mathematical curve).

In the scenario described—where latency remains flat up to a traffic threshold and then spikes exponentially—the relationship is strictly monotonic (as traffic increases, latency never decreases), but heavily non-linear. This structural difference explains why Pearson struggles while Spearman succeeds.

**Why Pearson Correlation Fails:**

The Pearson correlation coefficient ($r$) evaluates how tightly data points cluster around a straight line:$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

- Violation of Linearity: Below the traffic threshold, increases in traffic produce zero change in latency (a flat slope of zero). Past the threshold, the relationship turns exponential (a steep, curving slope). A straight line is a poor approximation for this hockey-stick shape.

- Distortion by Extreme Outliers: The exponential spike creates massive values in the upper right tail of the dataset. These points exert disproportionately high leverage on Pearson's covariance calculation, pulling the best-fit line away from the bulk of the data and degrading the overall score.

- Flattened Correlation Score: Because Pearson expects a constant, proportional rate of change, the combination of a flat phase and a rapid exponential curve results in a dampened $r$ value that fails to reflect how tightly coupled the two variables actually are.

**Why Spearman Rank Correlation Succeeds**

The Spearman rank correlation coefficient ($\rho$) bypasses raw values entirely. It converts both variables into ranks (ordering them from smallest to largest) and then calculates the Pearson correlation on those ranks:$$\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

- Transformation to Monotonicity: By replacing raw traffic and latency values with their ranks, Spearman strips away the geometric distortion of the exponential spike. An exponential jump from 50ms to 5,000ms is transformed into a clean, ordinal step (e.g., moving from the 80th rank to the 99th rank).

- Capturing Strict Ordering: Because the system exhibits a reliable ordering—higher traffic always yields equal or higher latency—the ranks of traffic and latency will align almost perfectly.

- Invariance to Scale: Spearman cares only whether an increase in $X$ corresponds to an increase in $Y$. Whether that increase is linear, logarithmic, or exponential does not matter; as long as the trajectory never reverses, Spearman will yield a high correlation coefficient close to 1.0.

### Guided Question:

If a dataset exhibits a strong positive linear correlation ($r = 0.85$), what happens to the Pearson correlation coefficient if a single extreme outlier is introduced in the bottom-right corner of the scatter plot?

### Answer:

When a single extreme outlier is introduced in the bottom-right corner of a scatter plot (where $X$ is very high and $Y$ is very low), the Pearson correlation coefficient ($r$) drops significantly, and can even flip to zero or become negative depending on the magnitude and leverage of that single point.

**The Mechanism Behind the Drop**

The Pearson correlation coefficient measures how closely data points align along a single straight line by calculating the normalized covariance between $X$ and $Y$:$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

Introducing an extreme outlier in the bottom-right corner distorts this calculation across three distinct mechanics:

1. **Negative Term in the Covariance Sum:**

For points that align with a positive trend ($r = 0.85$), when $x_i > \bar{x}$, $y_i$ is also typically $> \bar{y}$, making the product $(x_i - \bar{x})(y_i - \bar{y})$ positive. A bottom-right outlier has a very large positive distance from the mean in $X$ ($(x_i - \bar{x}) \gg 0$) but a large negative distance from the mean in $Y$ ($(y_i - \bar{y}) \ll 0$). Their product is a massive negative number, directly subtracting from the numerator sum.

2. **Extreme Leverage:**

Because the point is located at an extreme value of $X$, it possesses high leverage. The squared deviation term $(x_i - \bar{x})^2$ in the denominator inflates drastically, reducing the magnitude of the overall ratio.

3. **Shifting Means:**


The extreme $X$ value pulls the overall sample mean $\bar{x}$ to the right, and the low $Y$ value pulls $\bar{y}$ downward, altering the mean deviations for every other point in the dataset.

### Guided Question:

When building a multiple linear regression model, data scientists check the correlation matrix of independent features. If two features (e.g., height in inches and height in centimeters) have a Pearson correlation of $r = 1.0$:What statistical problem does this introduce into linear regression coefficient estimation, and how is it resolved?

When two features have a Pearson correlation of $r = 1.0$, they exhibit perfect multicollinearity. They carry the exact same mathematical information scaled by a constant factor (e.g., $\text{cm} = \text{inches} \times 2.54$).

**The Statistical Problem: Matrix Singularity** 

In Ordinary Least Squares (OLS) linear regression, the vector of regression coefficients ($\hat{\beta}$) is calculated using the normal equation:

To compute this, the matrix $X^T X$ (where $X$ is the matrix of independent features) must be invertible.

- Singular Matrix: When two columns are perfectly correlated ($r = 1.0$), the columns of $X$ are linearly dependent. This causes the matrix $X^T X$ to become singular (its determinant is zero).

- Non-Invertibility: A singular matrix cannot be inverted ($(X^T X)^{-1}$ does not exist).

- Consequences: The OLS algorithm breaks down completely. Mathematically, there is no unique solution; instead, there are infinitely many combinations of coefficients that achieve the exact same minimum error. Furthermore, the standard errors of the coefficients approach infinity, rendering them entirely unstable and uninterpretable.

**How It Is Resolved**

Because perfect multicollinearity makes parameter estimation impossible, data scientists must eliminate the linear dependency before fitting the model:

- **Feature Dropping (Redundancy Removal):**
Since perfectly correlated features provide duplicate information, the standard and most direct solution is to drop one of them entirely (e.g., keep height in inches and discard height in centimeters). This restores linear independence and makes $X^T X$ invertible.

- **L2 Regularization (Ridge Regression):**
If moving to a regularized model like Ridge Regression, a small tuning parameter ($\lambda$) is added to the diagonal of the $X^T X$ matrix before inversion ($X^T X + \lambda I$). This ensures the matrix is invertible even in the presence of strong or perfect multicollinearity, distributing the coefficient weights between the redundant features.